# AutoSelect and AutoVAR

This notebook covers two advanced automatic model selection tools:

1. **AutoSelect**: compares models across different families (ARIMA, ETS, Theta, baselines)
   and selects the best performer using temporal cross-validation.
2. **AutoVAR**: automatic Vector Autoregression for multivariate time series — selects
   the optimal lag order and performs diagnostic checks.

**Topics covered:**
- AutoSelect: comparing ARIMA, ETS, and Theta on univariate series
- Cross-validation-based model selection
- AutoVAR: automatic lag selection for multivariate models
- VAR diagnostics: stability, Granger causality
- Multivariate forecasting with fan charts

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecastbox.auto import AutoSelect, AutoVAR, AutoARIMA, AutoETS

import sys
sys.path.insert(0, "..")
from utils.helpers import load_m3_sample, load_airline, get_series

# Load macro_brazil from basic_forecasting examples
macro_path = "../../basic_forecasting/data/macro_brazil.csv"
df_macro = pd.read_csv(macro_path, parse_dates=["date"], index_col="date")

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)

print("Macro Brazil columns:", df_macro.columns.tolist())
print("Shape:", df_macro.shape)
df_macro.head()

## 1. AutoSelect

**AutoSelect** compares multiple model families and selects the best one. Instead of relying
on in-sample criteria (AIC/BIC), it uses **temporal cross-validation** to evaluate out-of-sample
forecast accuracy.

Supported model families: `arima`, `ets`, `naive`, `snaive`, `drift`

The comparison metric can be RMSE, MAE, MAPE, or MASE. Let's apply AutoSelect to Brazilian
GDP growth:

In [ ]:
# Apply AutoSelect to Brazilian GDP growth
gdp = df_macro["gdp_growth"].dropna()

print(f"GDP Growth series: {len(gdp)} observations")
print(f"Date range: {gdp.index[0]} to {gdp.index[-1]}")

# AutoSelect compares ARIMA, ETS, and Theta
auto_select = AutoSelect(
    families=["arima", "ets", "naive", "snaive", "drift"],
    cv_type="expanding",
    cv_horizon=12,
    cv_step=1,
    metric="rmse",
)
result = auto_select.fit(gdp, m=12)

print(f"\nBest model family: {result.best_family}")
print(f"Best model: {result.best_model_name}")
print(f"\n{result.summary()}")

## 2. AutoSelect with Cross-Validation

The ranking is based on temporal cross-validation scores. Let's examine the full ranking
and visualize the comparison across model families.

In [ ]:
# Show the full ranking table
print("Model Ranking by CV RMSE:")
print(result.ranking.to_string())

# Visualize the comparison
fig, ax = plt.subplots(figsize=(10, 5))
result.plot_comparison(ax=ax, title="AutoSelect — Model Comparison (CV RMSE)")
plt.tight_layout()
plt.show()

# Show cross-validation scores for each family
print("\nCV Scores per family:")
for family, scores in result.all_cv_results.items():
    scores_arr = np.array(scores)
    print(f"  {family:8s}: mean={scores_arr.mean():.4f}, std={scores_arr.std():.4f}, "
          f"min={scores_arr.min():.4f}, max={scores_arr.max():.4f}")

## 3. AutoVAR

**Vector Autoregression (VAR)** extends AR models to the multivariate setting. A VAR(p) model
for $k$ variables is:

$$\mathbf{y}_t = \mathbf{c} + \mathbf{A}_1 \mathbf{y}_{t-1} + \mathbf{A}_2 \mathbf{y}_{t-2} + \cdots + \mathbf{A}_p \mathbf{y}_{t-p} + \mathbf{u}_t$$

**AutoVAR** automatically selects the optimal lag order $p$ by fitting VAR models for
$p = 1, 2, \ldots, p_{\max}$ and selecting the one that minimizes AIC or BIC.

Let's build a VAR for Brazilian GDP growth and inflation:

In [ ]:
# Prepare multivariate data: GDP growth and inflation
var_data = df_macro[["gdp_growth", "inflation"]].dropna()

print(f"VAR data shape: {var_data.shape}")
print(f"Variables: {var_data.columns.tolist()}")

# Visualize the two series
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
var_data["gdp_growth"].plot(ax=axes[0], color="steelblue", title="GDP Growth")
axes[0].set_ylabel("GDP Growth (%)")
axes[0].grid(True, alpha=0.3)

var_data["inflation"].plot(ax=axes[1], color="darkorange", title="Inflation")
axes[1].set_ylabel("Inflation (%)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Fit AutoVAR with automatic lag selection
auto_var = AutoVAR(max_lags=12, ic="bic", trend="c")
var_result = auto_var.fit(var_data)

print(f"\nSelected lag order: {var_result.selected_lag}")
print(f"Number of variables: {var_result.n_vars}")
print(f"IC criterion: {var_result.ic_name}")

# Show IC table for all lag orders
print("\nInformation Criterion by Lag Order:")
print(var_result.ic_table.to_string())

## 4. VAR Diagnostics

A well-specified VAR model should satisfy:

1. **Stability**: all eigenvalues of the companion matrix lie inside the unit circle
   (|eigenvalue| < 1). If not, the system is explosive.
2. **Granger causality**: tests whether one variable helps predict another beyond its own lags.
   $H_0$: variable $x$ does not Granger-cause variable $y$.

In [ ]:
# VAR Diagnostics
print(var_result.summary())

# 1. Stability check — eigenvalues of the companion matrix
var_model = var_result.model
eigenvalues = np.abs(np.linalg.eigvals(var_model.coefs.reshape(-1, var_result.n_vars).T
                                        if var_result.selected_lag == 1
                                        else np.vstack([
                                            var_model.coefs.reshape(var_result.selected_lag * var_result.n_vars,
                                                                     var_result.n_vars).T,
                                            np.eye(var_result.n_vars * (var_result.selected_lag - 1),
                                                   var_result.n_vars * var_result.selected_lag)
                                        ])))

# Use the built-in stability check from statsmodels
is_stable = var_model.is_stable()
print(f"\nStability check: {'STABLE' if is_stable else 'UNSTABLE'}")
print(f"All eigenvalues inside unit circle: {is_stable}")

# Plot eigenvalues on unit circle
fig, ax = plt.subplots(figsize=(6, 6))
roots = var_model.roots
theta = np.linspace(0, 2 * np.pi, 100)
ax.plot(np.cos(theta), np.sin(theta), "k--", alpha=0.3, label="Unit circle")
ax.scatter(np.real(roots), np.imag(roots), color="steelblue", s=80, zorder=5, label="Eigenvalues")
ax.set_xlabel("Real")
ax.set_ylabel("Imaginary")
ax.set_title("VAR Stability — Eigenvalues")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2. Granger causality tests
print("\nGranger Causality Tests:")
for col in var_data.columns:
    test_result = var_model.test_causality(col, causing=var_data.columns.drop(col).tolist())
    print(f"\n  {var_data.columns.drop(col).tolist()} → {col}:")
    print(f"    Test statistic: {test_result.test_statistic:.4f}")
    print(f"    p-value: {test_result.pvalue:.4f}")
    if test_result.pvalue < 0.05:
        print(f"    → Reject H0: Granger causality detected at 5% level")
    else:
        print(f"    → Fail to reject H0: No Granger causality at 5% level")

## 5. VAR Forecasting

VAR produces **multivariate forecasts** — simultaneous predictions for all variables in the
system. We can visualize these as individual series with prediction intervals (fan charts).

In [ ]:
# Multivariate forecast
h = 12
fc = var_result.forecast(h=h, level=(80, 95))
fc_df = fc.to_dataframe()

print(f"Forecast horizon: {h} periods")
print(f"Forecast shape: {fc_df.shape}")
print("\nForecast (first 6 periods):")
print(fc_df.head(6).to_string())

# Fan chart: plot forecast for each variable
variables = var_data.columns.tolist()
colors = ["steelblue", "darkorange"]

fig, axes = plt.subplots(len(variables), 1, figsize=(12, 5 * len(variables)), sharex=True)
if len(variables) == 1:
    axes = [axes]

fc_index = pd.date_range(start=var_data.index[-1] + pd.DateOffset(months=1), periods=h, freq="MS")

for i, (var_name, color) in enumerate(zip(variables, colors)):
    ax = axes[i]

    # Historical data
    ax.plot(var_data.index, var_data[var_name].values, color=color, linewidth=1.5, label="Historical")

    # Point forecast
    ax.plot(fc_index, fc.point[:, i] if fc.point.ndim > 1 else fc.point,
            color="red", linewidth=2, label="Forecast")

    # 95% prediction interval
    if fc.lower_95 is not None and fc.upper_95 is not None:
        lower = fc.lower_95[:, i] if fc.lower_95.ndim > 1 else fc.lower_95
        upper = fc.upper_95[:, i] if fc.upper_95.ndim > 1 else fc.upper_95
        ax.fill_between(fc_index, lower, upper, alpha=0.15, color="red", label="95% PI")

    # 80% prediction interval
    if fc.lower_80 is not None and fc.upper_80 is not None:
        lower = fc.lower_80[:, i] if fc.lower_80.ndim > 1 else fc.lower_80
        upper = fc.upper_80[:, i] if fc.upper_80.ndim > 1 else fc.upper_80
        ax.fill_between(fc_index, lower, upper, alpha=0.3, color="red", label="80% PI")

    ax.set_title(f"VAR Forecast — {var_name}")
    ax.set_ylabel(var_name)
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Exercise 1: Use AutoSelect on all M3 monthly series — SOLUTION

Apply AutoSelect to each monthly series in the M3 sample dataset. For each series, report
which model family was selected as best. Is there a single family that dominates?

In [ ]:
# Exercise 1 — SOLUTION: AutoSelect on all 5 M3 monthly series
# AutoSelect uses temporal cross-validation to compare model families,
# so the best family may differ across series depending on data characteristics.

# Step 1: Load M3 data and identify monthly series
m3 = load_m3_sample()
monthly_ids = m3[m3["frequency"] == "monthly"]["series_id"].unique()
print(f"Monthly series to evaluate: {monthly_ids.tolist()}")
print(f"Number of series: {len(monthly_ids)}")
print()

# Step 2: Apply AutoSelect to each monthly series
select_results = []
for sid in monthly_ids:
    y = get_series(m3, sid)
    print(f"Fitting AutoSelect for '{sid}' ({len(y)} obs)...")

    auto_sel = AutoSelect(
        families=["arima", "ets", "naive", "snaive", "drift"],
        cv_type="expanding",
        cv_horizon=12,
        cv_step=1,
        metric="rmse",
    )
    res = auto_sel.fit(y, m=12)

    select_results.append({
        "Series": sid,
        "Best Family": res.best_family,
        "Best Model": res.best_model_name,
        "CV RMSE (mean)": round(np.array(res.all_cv_results[res.best_family]).mean(), 4),
        "N Families Tested": len(res.ranking),
    })

# Step 3: Display summary table
results_df = pd.DataFrame(select_results)
print("\n" + "=" * 80)
print("AutoSelect Results — M3 Monthly Series")
print("=" * 80)
print(results_df.to_string(index=False))
print("=" * 80)

# Step 4: Analyze which family dominates
family_counts = results_df["Best Family"].value_counts()
print(f"\nFamily selection frequency:")
for family, count in family_counts.items():
    print(f"  {family}: {count}/{len(monthly_ids)} series ({100*count/len(monthly_ids):.0f}%)")

if family_counts.iloc[0] == len(monthly_ids):
    print(f"\n→ {family_counts.index[0]} dominates all series.")
else:
    print(f"\n→ No single family dominates. The best model depends on the data characteristics.")
    print("  This is why AutoSelect is valuable — it adapts to each series.")

## Exercise 2: Build AutoVAR for US macro (GDP + CPI + unemployment) — SOLUTION

Load the US macroeconomic dataset. Build an AutoVAR model with GDP growth, CPI inflation,
and unemployment. Check stability and Granger causality, then produce an 8-quarter-ahead forecast.

In [ ]:
# Exercise 2 — SOLUTION: AutoVAR for US macro (GDP, CPI, unemployment)
# We build a trivariate VAR model for the US economy and analyze the
# dynamic interactions between GDP growth, inflation, and unemployment.

# Step 1: Load US macro data
df_us = pd.read_csv(
    "../../basic_forecasting/data/macro_us.csv",
    parse_dates=["date"],
    index_col="date",
)
print(f"US Macro data shape: {df_us.shape}")
print(f"Columns: {df_us.columns.tolist()}")
print(f"Date range: {df_us.index[0]} to {df_us.index[-1]}")

# Step 2: Select the 3 variables of interest
var_cols = ["gdp_growth", "cpi_inflation", "unemployment"]
us_var_data = df_us[var_cols].dropna()
print(f"\nVAR data shape: {us_var_data.shape}")
print(f"Variables: {us_var_data.columns.tolist()}")

# Step 3: Visualize the three series
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
colors = ["steelblue", "darkorange", "forestgreen"]
titles = ["GDP Growth (%)", "CPI Inflation (%)", "Unemployment Rate (%)"]

for i, (col, color, title) in enumerate(zip(var_cols, colors, titles)):
    us_var_data[col].plot(ax=axes[i], color=color, linewidth=1.2)
    axes[i].set_title(title)
    axes[i].set_ylabel(col)
    axes[i].grid(True, alpha=0.3)

plt.suptitle("US Macroeconomic Variables", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Step 4: Fit AutoVAR with automatic lag selection
auto_var = AutoVAR(max_lags=12, ic="bic", trend="c")
var_result = auto_var.fit(us_var_data)

print(f"\n--- AutoVAR Results ---")
print(f"Selected lag order: {var_result.selected_lag}")
print(f"Number of variables: {var_result.n_vars}")
print(f"IC criterion: {var_result.ic_name}")

# Show IC table for all lag orders
print(f"\nInformation Criterion by Lag Order:")
print(var_result.ic_table.to_string())

# Step 5: Stability check
var_model = var_result.model
is_stable = var_model.is_stable()
print(f"\n--- Stability Check ---")
print(f"VAR is stable: {is_stable}")
print(f"All eigenvalues inside unit circle: {is_stable}")

if is_stable:
    print("→ The system is stable (stationary). Forecasts will converge to the mean.")
else:
    print("→ WARNING: The system is unstable. Long-horizon forecasts may be unreliable.")

# Plot eigenvalues on unit circle
fig, ax = plt.subplots(figsize=(6, 6))
roots = var_model.roots
theta = np.linspace(0, 2 * np.pi, 100)
ax.plot(np.cos(theta), np.sin(theta), "k--", alpha=0.3, label="Unit circle")
ax.scatter(np.real(roots), np.imag(roots), color="steelblue", s=80, zorder=5, label="Eigenvalues")
ax.set_xlabel("Real")
ax.set_ylabel("Imaginary")
ax.set_title("VAR Stability — Eigenvalues on Unit Circle")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Step 6: Granger causality tests
# Test whether each variable Granger-causes the others
print("\n--- Granger Causality Tests ---")
print("(H0: causing variable does NOT Granger-cause the dependent variable)")

granger_results = []
for dep_var in var_cols:
    causing_vars = [c for c in var_cols if c != dep_var]
    for cause_var in causing_vars:
        test_result = var_model.test_causality(dep_var, causing=cause_var)
        sig = "***" if test_result.pvalue < 0.01 else "**" if test_result.pvalue < 0.05 else "*" if test_result.pvalue < 0.10 else ""
        granger_results.append({
            "Causing": cause_var,
            "Dependent": dep_var,
            "Test Stat": round(test_result.test_statistic, 4),
            "p-value": round(test_result.pvalue, 4),
            "Significant": sig,
        })
        status = "REJECT H0" if test_result.pvalue < 0.05 else "FAIL TO REJECT"
        print(f"  {cause_var} → {dep_var}: stat={test_result.test_statistic:.4f}, "
              f"p={test_result.pvalue:.4f} [{status}]")

granger_df = pd.DataFrame(granger_results)
print(f"\nGranger Causality Summary Table:")
print(granger_df.to_string(index=False))
print("\nSignificance: *** p<0.01, ** p<0.05, * p<0.10")

# Step 7: Multivariate forecast h=8 quarters
h = 8
fc = var_result.forecast(h=h, level=(80, 95))
fc_df = fc.to_dataframe()

print(f"\n--- Multivariate Forecast (h={h}) ---")
print(fc_df.to_string())

# Step 8: Fan charts for each variable
fig, axes = plt.subplots(len(var_cols), 1, figsize=(12, 5 * len(var_cols)), sharex=True)
colors_fc = ["steelblue", "darkorange", "forestgreen"]

# Create forecast index
fc_index = pd.date_range(
    start=us_var_data.index[-1] + pd.DateOffset(months=1),
    periods=h,
    freq="MS",
)

for i, (col, color) in enumerate(zip(var_cols, colors_fc)):
    ax = axes[i]

    # Historical data (last 60 periods for readability)
    hist_data = us_var_data[col].iloc[-60:]
    ax.plot(hist_data.index, hist_data.values, color=color, linewidth=1.5, label="Historical")

    # Point forecast
    point = fc.point[:, i] if fc.point.ndim > 1 else fc.point
    ax.plot(fc_index, point, color="red", linewidth=2, label="Forecast")

    # 95% prediction interval
    if fc.lower_95 is not None and fc.upper_95 is not None:
        lower = fc.lower_95[:, i] if fc.lower_95.ndim > 1 else fc.lower_95
        upper = fc.upper_95[:, i] if fc.upper_95.ndim > 1 else fc.upper_95
        ax.fill_between(fc_index, lower, upper, alpha=0.15, color="red", label="95% PI")

    # 80% prediction interval
    if fc.lower_80 is not None and fc.upper_80 is not None:
        lower = fc.lower_80[:, i] if fc.lower_80.ndim > 1 else fc.lower_80
        upper = fc.upper_80[:, i] if fc.upper_80.ndim > 1 else fc.upper_80
        ax.fill_between(fc_index, lower, upper, alpha=0.3, color="red", label="80% PI")

    ax.set_title(f"VAR Forecast — {col}")
    ax.set_ylabel(col)
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)

plt.suptitle(f"US Macro VAR({var_result.selected_lag}) — {h}-Step Forecast", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(f"\n--- Summary ---")
print(f"VAR({var_result.selected_lag}) model for {var_cols}")
print(f"Stable: {is_stable}")
print(f"Forecast horizon: {h} periods")
print(f"Significant Granger causality relationships:")
sig_granger = granger_df[granger_df["p-value"] < 0.05]
if len(sig_granger) > 0:
    for _, row in sig_granger.iterrows():
        print(f"  {row['Causing']} → {row['Dependent']} (p={row['p-value']:.4f})")
else:
    print("  None at 5% significance level")